In [ ]:
"""
<FAISS 벡터 DB>
 - 페이스북에서 만든 벡터 DB
 - 대용량 데이터 검색에 주로 사용됨
 - GPU 연산에 최적화 되어 있음
"""

#### 기본 라이브러리 불러들이기

In [ ]:
### PDF 문서 로더 라이브러리
from langchain_community.document_loaders import PyPDFLoader

### 텍스트 분할기 : 재귀적 문자 분할기
from langchain_text_splitters import RecursiveCharacterTextSplitter

### 임베딩 처리 모델(문자를 숫자로)
from langchain_ollama.embeddings import OllamaEmbeddings

### 벡터 DB
from langchain_community.vectorstores import FAISS

### 벡터 DB(크로마)
from langchain_chroma import Chroma

### PDF 문서 로드(읽어들이기)

In [ ]:
"""
- 사용 문서 : Data 폴더 내에 pdf 파일
- PDF 문서 로드하기
- 재귀적 텍스트 분할기로 분할하기
    -> 분할 청크 1000자, 중복 기준 200자
- 분할 후 변수명  : texts

- 청크(문서) 갯수 출력
"""

In [15]:
### pdf 문서 경로 변수 설정
file_path = "./Data/2024_KB_부동산_보고서_최종.pdf"

### 로더 생성하기
loader = PyPDFLoader(file_path)
loader

### 파일 불러들이기 변수명 document
document = loader.load()
document

### 분할기 클래스 생성
text_spitter = RecursiveCharacterTextSplitter(
    chunk_size = 1000,
    chunk_overlap = 200
)
text_spitter


### 분할하기
texts = text_spitter.split_documents(document)

print(f"청크(문서) 갯수 출력 : {len(texts)}")

청크(문서) 갯수 출력 : 135


## FAISS

In [ ]:
"""
<처리 순서>
1. 임베딩 클래스 정의 : 임베딩 전용 모델 정의
2. 파이스(FAISS) DB 저장할 위치(폴더) 정의 : "./FAISS_DB/"
3. 파이스(FAISS) DB 클래스 생성 : FAISS.from_documents() 사용
4. 파이스(FAISS) DB에 저장 시키기 : save_local(저장할 위치) 함수 사용
"""

In [18]:
# 1. 임베딩 클래스 정의 : 임베딩 전용 모델 정의(nomic-embed-text)
embedding_function = OllamaEmbeddings(model="nomic-embed-text")
embedding_function

# 2. 파이스(FAISS) DB 저장할 위치(폴더) 정의 : "./FAISS_DB/"
faiss_save_path = "./FAISS_DB/"

# 3. 파이스(FAISS) DB 클래스 생성 : FAISS.from_documents() 사용
vectordb = FAISS.from_documents(
    ### 청크(문서) 원본 데이터(분할한 데이터)
    documents = texts,

    ### 임베딩 전용 모델 정의
    embedding = embedding_function

    # - FAISS 는 별도로 저장하기가 없어 아래와 같이 함수 써야 함
)

# 4. 파이스(FAISS) DB에 저장 시키기 : save_local(저장할 위치) 함수 사용
vectordb.save_local(faiss_save_path)

### 파이스(FAISS) DB에서 데이터 불러들이기(서비스시에 사용)

In [ ]:
"""
<처리 순서>
  1. 임베딩 모델 클래스 정의 : 임베딩 전용 모델 정의
  2. 파이스(FAISS) DB가 위치한 디렉토리 정의
  3. 파이스(FAISS) 클래스 생성 : 데이터 불러들임
"""

In [21]:
#  1. 임베딩 모델 클래스 정의 : 임베딩 전용 모델 정의
embedding_function = OllamaEmbeddings(model="nomic-embed-text")

#  2. 파이스(FAISS) DB가 위치한 디렉토리 정의
faiss_path = "./FAISS_DB/"

#  3. 파이스(FAISS) 클래스 생성 : 데이터 불러들임(load_local 함수 사용)
vectordb = FAISS.load_local(
    embeddings = embedding_function,
    folder_path = faiss_path,

    ### 보안 경고 무시
    # - FAISS는 실행 시 보안 경고가 자주 출력됨
    allow_dangerous_deserialization = True
)

vectordb

### 불러들인 FAISS DB 데이터를 이용하여 검색 테스트하기

In [22]:
"""
<벡터 데이터베이스에서 검색 방법>
 * 벡터 데이터베이스는 클래스로 검색 함수를 제공하고 있음
 * 검색 함수
   - similarity_search(질문, k=2)
     : 단순히 유사성 검색을 통해 즉각적인 결과를 확인하고자 할 때 사용
       (RAG 통합 서비스시에는 사용 못함, 테스트용으로 사용)
     : 질문 -> 사용자가 요청에 사용할 입력 받은 문장을 의미함
     : k   -> 질문과 유사한 조회 갯수(조회 청크 갯수)


   - as_retriver(search_kwargs={"k" : n})
     : 랭체인(lang chain) 파이프라인(랭체인) 연결을 통해 **"RAG 통합 서비스시에 사용"** 됨
     : 랭체인 파이프라인(랭체인)으로 연결시 사용하는 "표준 함수"임
     : search_kwargs={"k" : n}
       -> 질문과 유사한 조회 갯수(조회 청크 갯수)
       -> 질문은 파이프라인(랭체인)에서 연결되어 사용되기에 별도로 검색 함수에 추가하지 않음
"""
"""
<처리 순서>
 1. 예시 질문 정의 : 입력 받아도 되며, 문자열로 정의해도 됨(테스트용)
 2. 검색 함수 정의 : 정의시 응답 갯수 정의 및 검색 결과 반환함
 3. 검색 결과 출력 : 응답 갯수가 여러개인 경우 반복문을 이용하여 응답 결과 출력 처리
"""

'\n<처리 순서>\n 1. 예시 질문 정의 : 입력 받아도 되며, 문자열로 정의해도 됨(테스트용)\n 2. 검색 함수 정의 : 정의시 응답 갯수 정의 및 검색 결과 반환함\n 3. 검색 결과 출력 : 응답 갯수가 여러개인 경우 반복문을 이용하여 응답 결과 출력 처리\n'

In [ ]:
"""
- similarity_search(질문, k=2)
- 질문 출력
- 응답 결과 2건 모두 출력
"""

In [67]:
### 질문 정의
question = input("질문을 입력")

answer = vectordb.similarity_search(question, k=2)

print(f"[질문] : {question}")

for i in range(0, 2, 1):
    print(f"[응답 결과 {i+1}번째] : {answer[i].page_content}")
    print("="*100)

[질문] : 부동산
[응답 결과 1번째] : 61 
2024 KB부동산 보고서: 상업용 부동산시장 진단과 전망 
 
무점포소매업의 경우 인터넷 쇼핑이 호조를 
보이면서 성장을 주도하고 있다. 홈쇼핑이나 
방문 배달 및 소매점은 횡보세를 보이나 인터
넷 쇼핑은 가파른 성장세가 멈추지 않고 있다. 
코로나팬데믹 이후 소비 패턴 변화로 온라인 
소비가 주요 소비 트렌드로 자리 잡은 셈이다. 
온라인 소비 문화가 정착되면서 판매 상품이 
다양해지고 있으며, 스마트폰을 통한 결제 서
비스의 편리함까지 더해지면서 시장 성장이 가
속화되고 있는 것으로 판단된다. 
 
 
 
■ 온라인시장 급성장에 따라 상권은 전반적으로 위축세 지속 
온라인시장이 가파르게 성장하면서 상가시장은 전반적으로 위축세가 지속되고 있다. 상가 임대료는 
코로나팬데믹이 끝난 이후에도 지속적으로 하락하고 있다. 대부분의 상가 규모에서 임대료지수가 하락
하고 있으며, 특히 소규모상가의 임대료 하락이 가파르게 나타나고 있다11. 대출 금리가 낮아지면서 투
자 수익률은 다소 회복되었으나 여전히 크게 낮으며, 상가 공실률이 높다는 점도 매우 부담스러운 상황
이다. 
 
그림Ⅳ-28. 상가 유형별 임대료 지수 추이  그림Ⅳ-29. 상가 유형별 투자 수익률 추이 
 
 
 
자료: 한국부동산원 
 
 
주: 수익률은 4분기 시간가중수익률 
자료: 한국부동산원 
                                                        
11 중대형 상가는 2021년 6월 기준 일반건축물대장 상의 주용도가 상가(제1, 2종 근린생활시설, 판매시설, 운동시설, 위락시설)이고 3층 이상이거나 연면적 
330㎡ 초과인 일반건축물, 소규모 상가는 일반건축물대장상의 주용도가 상가이고 2층 이하이며 연면적 330㎡ 이하인 일반건축물, 집합상가는 집합건축물대
장상의 주용도가 상가인 집합건축물을 말한다. 
 그림Ⅳ-27. 무점포소매업 판매액 지수 추이 
 
 
 자료: 통계청
[응답 결과 2번째] : 3

#### 실습

In [ ]:
"""
* 사용자로부터 입력 받아서 처리
- (1). 사용자로부터 "Chroma_DB"를 사용할지 "FAISS_DB"를 사용할지 번호로 선택하게 함
    1번 : Chroma DB
    2번 : FAISS DB
    0번 : 반복 종료

- (2). (1)번에서 선택된 번호에 대한 검색을 진행시키기
  -> 사용자로부터 질문 입력 받기
  -> 답변 1개 출력하기
  -> 반복 질문 및 답변 수행 (종료 처리)
  -> (2)번 종료시 (1)번 DB 선택으로 이동하여 반복 진행
"""

In [75]:
### Chroma_DB에 대한 불러오기 함수
def chroma_db():    
    # 크로마(Chroma) 클래스 생성 :
    vectordb = Chroma(
    # 임베딩 모델 정의
    embedding_function = embedding_function,
    # 불러들일 크로마 DB 폴더 위치 정의
    persist_directory = DB_dict[choose]
    )
    return vectordb

### FAISS_DB에 대한 불러오기 함수
def faiss_db():    
    vectordb = FAISS.load_local(
    embeddings = embedding_function,
    folder_path = faiss_path,
    ### 보안 경고 무시
    # - FAISS는 실행 시 보안 경고가 자주 출력됨
    allow_dangerous_deserialization = True
    )
    return vectordb

In [78]:
### DB 별 딕셔너리
DB_dict = {"1번" : "Chroma_DB", "2번" : "FAISS_DB", "0번" : "반복 종료"}

chroma = chroma_db()
faiss = faiss_db()
while True:
    ### 사용자에게 입력 받을 변수 하나 정하기
    choose = input("선택할 번호를 입력하세요. 예시) 1번")

    ### 임베딩 정의(둘 다 똑같다.)
    embedding_function = OllamaEmbeddings(model="nomic-embed-text")

    ### (조건문)
    if choose == "1번":
        chroma
    elif choose == "2번":
        faiss
    elif choose == "0번":
        break
    else :
        print("정확한 번호를 입력하세요!")
        continue
    
    
       
    ##### 위에서 chroma냐 faiss냐 정해진 vectordb가 내려와서 실행됨.
    ### 질문 변수 정의
    question = input("검색어를 입력하세요. 예시) 부동산")
    ### 답변 변수 정의
    answer = vectordb.similarity_search(question, k=1)
    ### 선택한 모델 출력
    print(f"[선택한 DB] : {DB_dict[choose]}")
    ### 질문 출력
    print(f"[질문] : {question}")
    ### 답변 출력
    print(f"[답변] : {answer[0].page_content}")

[선택한 DB] : Chroma_DB
[질문] : 부동산
[답변] : 61 
2024 KB부동산 보고서: 상업용 부동산시장 진단과 전망 
 
무점포소매업의 경우 인터넷 쇼핑이 호조를 
보이면서 성장을 주도하고 있다. 홈쇼핑이나 
방문 배달 및 소매점은 횡보세를 보이나 인터
넷 쇼핑은 가파른 성장세가 멈추지 않고 있다. 
코로나팬데믹 이후 소비 패턴 변화로 온라인 
소비가 주요 소비 트렌드로 자리 잡은 셈이다. 
온라인 소비 문화가 정착되면서 판매 상품이 
다양해지고 있으며, 스마트폰을 통한 결제 서
비스의 편리함까지 더해지면서 시장 성장이 가
속화되고 있는 것으로 판단된다. 
 
 
 
■ 온라인시장 급성장에 따라 상권은 전반적으로 위축세 지속 
온라인시장이 가파르게 성장하면서 상가시장은 전반적으로 위축세가 지속되고 있다. 상가 임대료는 
코로나팬데믹이 끝난 이후에도 지속적으로 하락하고 있다. 대부분의 상가 규모에서 임대료지수가 하락
하고 있으며, 특히 소규모상가의 임대료 하락이 가파르게 나타나고 있다11. 대출 금리가 낮아지면서 투
자 수익률은 다소 회복되었으나 여전히 크게 낮으며, 상가 공실률이 높다는 점도 매우 부담스러운 상황
이다. 
 
그림Ⅳ-28. 상가 유형별 임대료 지수 추이  그림Ⅳ-29. 상가 유형별 투자 수익률 추이 
 
 
 
자료: 한국부동산원 
 
 
주: 수익률은 4분기 시간가중수익률 
자료: 한국부동산원 
                                                        
11 중대형 상가는 2021년 6월 기준 일반건축물대장 상의 주용도가 상가(제1, 2종 근린생활시설, 판매시설, 운동시설, 위락시설)이고 3층 이상이거나 연면적 
330㎡ 초과인 일반건축물, 소규모 상가는 일반건축물대장상의 주용도가 상가이고 2층 이하이며 연면적 330㎡ 이하인 일반건축물, 집합상가는 집합건축물대
장상의 주용도가 상가인 집합건축물을 말한다. 
 그림Ⅳ-27. 무점포소매업 판매액 지수 추이 
 
 
 자료: 통계청


In [37]:
DB_dict = {"1번" : "Chroma_DB", "2번" : "FAISS_DB", "0번" : "반복 종료"}

In [38]:
DB_dict["1번"]

'Chroma_DB'